# Association Rule Mining: Reproduction-Ready Market Basket Experiment

**CMPE 255 — Assignment 1, Part 2**

This notebook presents an execution-ready experiment inspired by the popular Kaggle Groceries transaction dataset. It deliberately contains **no executed, dataset-dependent results**. When the source CSV is absent, every dependent stage is skipped safely and actionable setup instructions are displayed.

**CRISP-DM map:** Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation → Deployment.

## 1. Business Understanding

### Objective
Discover products that co-occur within shopping transactions so that a retailer can form hypotheses for cross-selling, product placement, bundles, and recommendation systems.

### Analytical goals
- Find frequent product combinations with Apriori.
- Generate rules of the form *antecedent → consequent*.
- evaluate rules using support, confidence, and lift.
- Rank interpretable candidates without claiming causal effects.

### Success criteria
Technical success means a validated, reproducible pipeline that yields non-empty, stable rules under documented thresholds when suitable data are supplied. Business success requires prospective validation (for example, an A/B test); association metrics alone do not establish business value.

## 2. Data Understanding

The expected Groceries-style file contains one product line per member-shopping-date event:

| Concept | Preferred column | Accepted aliases |
|---|---|---|
| Customer/member | `Member_number` | `member_number`, `member`, `customer_id` |
| Shopping date | `Date` | `date`, `transaction_date` |
| Product | `itemDescription` | `itemdescription`, `item`, `product`, `product_name` |

A basket is keyed by **member plus date**, not member alone. The member identifier is not used as a predictive feature. The loader also provides clear guidance when no data file is available.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")

BASE_DIR = Path.cwd()
if not (BASE_DIR / "association_rule_mining.ipynb").exists():
    candidate = BASE_DIR / "Assignment_1_Part_2" / "03_Association_Rule_Mining"
    if candidate.exists():
        BASE_DIR = candidate

DEFAULT_DATASET = BASE_DIR / "data" / "Groceries_dataset.csv"
DATASET_PATH = Path(os.environ.get("ARM_DATASET_PATH", DEFAULT_DATASET)).expanduser()
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.20
MIN_LIFT = 1.00
TOP_N_RULES = 20

print(f"Dataset path: {DATASET_PATH}")
print(f"Thresholds: support={MIN_SUPPORT}, confidence={MIN_CONFIDENCE}, lift={MIN_LIFT}")

In [ ]:
DATA_AVAILABLE = DATASET_PATH.is_file()
raw = pd.DataFrame()

if DATA_AVAILABLE:
    raw = pd.read_csv(DATASET_PATH)
    print(f"Loaded {raw.shape[0]:,} rows and {raw.shape[1]:,} columns.")
else:
    print(
        "DATASET NOT FOUND — analysis stages will be skipped safely.\n"
        "Provide a licensed Groceries-style CSV at:\n"
        f"  {DEFAULT_DATASET}\n"
        "or set ARM_DATASET_PATH to an absolute CSV path, then restart and run all cells.\n"
        "Expected concepts: member/customer ID, transaction date, and item description."
    )

raw.head() if DATA_AVAILABLE else pd.DataFrame({
    "status": ["Not executed"],
    "reason": ["Dataset unavailable; no sample rows or results are fabricated."]
})

## 3. Exploratory Data Analysis

The next cells inspect structure, types, cardinality, and the most common products **only after real data are loaded**. Counts shown after execution are descriptive, not evidence of association.

In [ ]:
if DATA_AVAILABLE:
    display(raw.info())
    display(raw.describe(include="all").T)
    display(raw.nunique(dropna=False).rename("unique_values").to_frame())
else:
    print("EDA skipped: provide the dataset using the instructions above.")

## 4–6. Data Preparation: Schema, Missing Values, Duplicates, and Cleaning

Cleaning decisions are explicit and auditable:

1. Resolve required columns case-insensitively from known aliases.
2. Parse dates; normalize product whitespace; reject blank products.
3. Report missingness before removing rows missing any basket key or product.
4. Report exact duplicate line items, then remove them so repeated identical rows do not distort binary baskets.
5. Keep member identifiers as strings to preserve leading zeros.

A duplicate product within the same basket is irrelevant for presence/absence association mining; quantities are outside this experiment's scope.

In [ ]:
ALIASES = {
    "member": ["member_number", "member", "customer_id"],
    "date": ["date", "transaction_date"],
    "item": ["itemdescription", "item", "product", "product_name"],
}

def resolve_columns(columns):
    normalized = {str(column).strip().lower(): column for column in columns}
    resolved = {}
    for concept, aliases in ALIASES.items():
        match = next((normalized[name] for name in aliases if name in normalized), None)
        if match is None:
            raise ValueError(
                f"Missing required {concept!r} column. Accepted names: {aliases}. "
                f"Available columns: {list(columns)}"
            )
        resolved[concept] = match
    return resolved

clean = pd.DataFrame(columns=["member", "date", "item", "transaction_id"])
column_map = None

if DATA_AVAILABLE:
    column_map = resolve_columns(raw.columns)
    working = raw[[column_map["member"], column_map["date"], column_map["item"]]].copy()
    working.columns = ["member", "date", "item"]
    working["member"] = working["member"].astype("string").str.strip()
    working["date"] = pd.to_datetime(working["date"], errors="coerce", dayfirst=True)
    working["item"] = working["item"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    working.loc[working["member"].eq(""), "member"] = pd.NA
    working.loc[working["item"].eq(""), "item"] = pd.NA

    missing_report = pd.DataFrame({
        "missing_count": working.isna().sum(),
        "missing_percent": working.isna().mean().mul(100),
    })
    duplicate_count = int(working.duplicated(subset=["member", "date", "item"]).sum())
    print("Missing-value analysis before row removal:")
    display(missing_report)
    print(f"Exact duplicate member-date-item rows before removal: {duplicate_count:,}")

    clean = working.dropna(subset=["member", "date", "item"]).drop_duplicates(
        subset=["member", "date", "item"]
    ).copy()
    clean["transaction_id"] = clean["member"] + "__" + clean["date"].dt.strftime("%Y-%m-%d")
    print(f"Clean rows: {len(clean):,}; transactions: {clean['transaction_id'].nunique():,}")
else:
    print("Missing-value, duplicate, and cleaning analyses skipped: no dataset was loaded.")

In [ ]:
if DATA_AVAILABLE and not clean.empty:
    item_counts = clean["item"].value_counts().head(20).sort_values()
    ax = item_counts.plot(kind="barh", figsize=(9, 6), color="#4472C4")
    ax.set(title="Most frequent products (line-item presence)", xlabel="Clean transaction-product rows", ylabel="Product")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "top_products.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Product-frequency visualization skipped: no cleaned transaction data are available.")

## 7–8. Basket Conversion and One-Hot Transaction Encoding

Each member-date transaction becomes a list of unique products. `TransactionEncoder` then creates a Boolean matrix: rows are transactions, columns are products, and `True` means that product occurs in that basket. This representation is required by `mlxtend.frequent_patterns.apriori`.

In [ ]:
transactions = []
basket = pd.DataFrame(dtype=bool)

if DATA_AVAILABLE and not clean.empty:
    transactions = clean.groupby("transaction_id", sort=True)["item"].agg(lambda values: sorted(set(values))).tolist()
    encoder = TransactionEncoder()
    encoded = encoder.fit(transactions).transform(transactions)
    basket = pd.DataFrame(encoded, columns=encoder.columns_, dtype=bool)
    print(f"Encoded basket shape: {basket.shape[0]:,} transactions × {basket.shape[1]:,} products")
    display(basket.head())
else:
    print("Basket conversion and one-hot encoding skipped: no cleaned transactions are available.")

## 9–10. Frequent Itemset Mining with Apriori

Apriori exploits downward closure: if an itemset is frequent, all of its subsets must also be frequent. `MIN_SUPPORT` controls candidate retention. For itemset \(X\):

\[
\mathrm{support}(X) = rac{\#\{	ext{transactions containing }X\}}{\#\{	ext{transactions}\}}
\]

No itemset or support is reported here until this cell runs on supplied data.

In [ ]:
frequent_itemsets = pd.DataFrame(columns=["support", "itemsets", "itemset_length"])

if not basket.empty:
    mined = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True, low_memory=True)
    if not mined.empty:
        mined["itemset_length"] = mined["itemsets"].map(len)
        frequent_itemsets = mined.sort_values(["support", "itemset_length"], ascending=[False, True]).reset_index(drop=True)
    print(f"Frequent itemsets found: {len(frequent_itemsets):,}")
    display(frequent_itemsets.head(20))
else:
    print("Apriori skipped: the one-hot basket is unavailable.")

## 11–14. Association Rules: Support, Confidence, and Lift

For a rule \(A \rightarrow B\):

- **Support**: \(P(A \cup B)\), the share of all baskets containing both sides.
- **Confidence**: \(P(B\mid A)=P(A\cup B)/P(A)\), how often the consequent appears when the antecedent appears.
- **Lift**: \(P(B\mid A)/P(B)\), observed co-occurrence relative to independence. Lift above 1 indicates positive association, equal to 1 independence, and below 1 negative association.

High confidence can merely reflect a very common consequent; lift adds a baseline adjustment. None of these metrics demonstrates causality.

In [ ]:
RULE_COLUMNS = [
    "antecedents", "consequents", "antecedent support", "consequent support",
    "support", "confidence", "lift", "leverage", "conviction",
]
rules = pd.DataFrame(columns=RULE_COLUMNS)

if not frequent_itemsets.empty and frequent_itemsets["itemsets"].map(len).max() >= 2:
    generated = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
    rules = generated.reset_index(drop=True)
    print(f"Rules meeting minimum confidence: {len(rules):,}")
    display(rules[RULE_COLUMNS].head(20))
elif not basket.empty:
    print("No rules generated: no frequent itemset contains at least two products at the chosen support threshold.")
else:
    print("Association-rule generation skipped: frequent itemsets are unavailable.")

## 15–16. Rule Filtering and Ranking Useful Rules

Candidate rules must meet all configured thresholds and have disjoint, nonempty sides (the generator already guarantees disjointness). Rules are ranked by lift, then confidence, then support. This is a transparent heuristic—not a claim of profit or actionability. Extremely rare rules may be unstable even with high lift, so support remains part of filtering and review.

In [ ]:
useful_rules = pd.DataFrame(columns=RULE_COLUMNS)

if not rules.empty:
    useful_rules = (
        rules.loc[
            (rules["support"] >= MIN_SUPPORT)
            & (rules["confidence"] >= MIN_CONFIDENCE)
            & (rules["lift"] >= MIN_LIFT)
        ]
        .sort_values(["lift", "confidence", "support"], ascending=False)
        .reset_index(drop=True)
    )
    useful_rules["antecedent_label"] = useful_rules["antecedents"].map(lambda x: ", ".join(sorted(x)))
    useful_rules["consequent_label"] = useful_rules["consequents"].map(lambda x: ", ".join(sorted(x)))
    print(f"Rules meeting all filters: {len(useful_rules):,}")
    display(useful_rules[["antecedent_label", "consequent_label", "support", "confidence", "lift"]].head(TOP_N_RULES))
elif not basket.empty:
    print("Rule filtering skipped: no rules met generation requirements.")
else:
    print("Rule filtering skipped: no dataset-derived rules exist.")

## 17. Visualization Code for Important Rules

The plots below are generated only from real rules. The scatter plot shows the support–confidence trade-off with lift encoded by color and size. The ranked bar chart makes the selected lift ordering explicit. Files are saved under `images/`; no placeholder result charts are created.

In [ ]:
if not useful_rules.empty:
    plot_rules = useful_rules.head(TOP_N_RULES).copy()
    fig, ax = plt.subplots(figsize=(9, 6))
    sizes = 60 + 140 * (plot_rules["lift"] / plot_rules["lift"].max())
    scatter = ax.scatter(
        plot_rules["support"], plot_rules["confidence"],
        c=plot_rules["lift"], s=sizes, cmap="viridis", alpha=0.8, edgecolor="black",
    )
    ax.set(title="Important association rules", xlabel="Support", ylabel="Confidence")
    fig.colorbar(scatter, ax=ax, label="Lift")
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "rule_scatter.png", dpi=150, bbox_inches="tight")
    plt.show()

    labels = plot_rules["antecedent_label"] + " → " + plot_rules["consequent_label"]
    fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(plot_rules))))
    ax.barh(labels.iloc[::-1], plot_rules["lift"].iloc[::-1], color="#70AD47")
    ax.axvline(1.0, color="black", linestyle="--", linewidth=1, label="Independence (lift = 1)")
    ax.set(title="Top filtered rules ranked by lift", xlabel="Lift", ylabel="Rule")
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "top_rules_by_lift.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Rule visualizations skipped: no executed, filtered rules are available.")

## 18. Interpretation of Product Relationships

After executing with real data, interpret each rule with all three metrics and the number/time span of baskets. A defensible template is:

> “Among the observed baskets, the antecedent and consequent occurred together at the reported support. The consequent occurred in the reported fraction of antecedent baskets (confidence), and their co-occurrence was the reported multiple of the independence baseline (lift). This is an association and does not imply that purchasing the antecedent causes purchasing the consequent.”

Check whether patterns persist across time windows, stores, and customer segments. This notebook makes **no product-specific interpretation** because the data were not executed.

## 19. Business Applications (Hypotheses, Not Findings)

Subject to validation, rules could inform:

- cross-sell recommendations and “frequently bought together” candidates;
- adjacent shelf placement or online navigation;
- bundle hypotheses and targeted coupons;
- coordinated inventory planning for co-occurring products.

Estimate margin, operational feasibility, and incremental impact before acting. Use randomized tests where possible; never describe an untested association as a proven commercial outcome.

## 20. Limitations

- The unavailable source data cannot be audited here for sampling, coverage, or provenance.
- Binary encoding ignores quantity, price, returns, and order sequence.
- A member-date key may merge separate same-day visits if no invoice identifier exists.
- Apriori can become expensive for low thresholds or many unique items.
- Results are threshold-sensitive and vulnerable to rare-item instability and multiple comparisons.
- Seasonality, promotions, availability, layout, and customer mix may confound co-occurrence.
- Support, confidence, and lift do not imply causality or profitability.
- Privacy and data-governance requirements must be checked before customer-level processing.

## 21. Future Improvements

- Prefer a true invoice/transaction identifier and add quantity/return handling.
- Compare FP-Growth with Apriori for scale and runtime.
- Tune thresholds using stability across temporal holdouts rather than convenience.
- Add rule redundancy pruning, statistical significance checks, and multiple-testing controls.
- Evaluate temporal, store, and customer-segment robustness while preserving privacy.
- Combine rules with margin and availability constraints.
- Validate interventions using controlled online or store experiments.
- Track drift and retrain under a documented deployment schedule.

## 22. CRISP-DM Conclusion

- **Business Understanding:** framed associative discovery as hypothesis generation for retail decisions.
- **Data Understanding:** documented the expected schema, unit of analysis, and provenance constraint.
- **Data Preparation:** implemented schema validation, missing/duplicate diagnostics, cleaning, transaction construction, and Boolean encoding.
- **Modeling:** implemented Apriori and association-rule generation with configurable thresholds.
- **Evaluation:** implemented filtering, ranking, visualization, and cautious interpretation using support, confidence, and lift.
- **Deployment:** identified candidate applications, validation requirements, monitoring needs, limitations, and improvements.

The pipeline is reproduction-ready, but dataset-dependent stages were intentionally not executed here. Therefore, **no frequent itemsets, metric values, rules, or business findings are claimed or fabricated**.